In [1]:
import os
import sys
import torch
import shutil
from ultralytics import YOLO
sys.path.append('../')
from project.yolo_training_utils import split_data_yolo


### Notebook Purpose
<b> This is notebook number 3a </b>

To give code guidance on training Yolo Classifier

### Notebook Order
1. getData
2. downloadData
3. trainResNetModel | trainPromptTransformerClassifier | trainViTClassifier | train YoloCLS
4. localMixtureEval

In [2]:
# Load a model
model = YOLO("yolov8n-cls.pt")  # build from YAML and transfer weights

#get data
dataDir = "../data/may_eval_dataset" ##where image data is saved. Note -> should be in their class folders


In [3]:
for i in os.listdir(dataDir):
    print(i, len(os.listdir(os.path.join(dataDir, i))))

X 5024
PG13 5039
PG 5011
R 5039
XXX 4993
images 3


In [4]:
# Get the number of CUDA devices available
num_devices = torch.cuda.device_count()

# Print information about each CUDA device
for i in range(num_devices):
    device_name = torch.cuda.get_device_name(i)
    print(f"Device {i}: {device_name}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Device 0: NVIDIA RTX A4000


## Process Data

In [5]:
# Create temporary directories for the split
topdir = ['train', 'val', 'test']
classdir = ['PG',"PG13", "R", "X", "XXX"]
for dir in topdir:
    os.makedirs(os.path.join(dataDir, 'images', dir), exist_ok=True)
    for cls in classdir:
        os.makedirs(os.path.join(dataDir, 'images', dir, cls), exist_ok=True)

In [6]:
train_list, val_list, test_list = split_data_yolo(dataDir, classes = ['PG', "PG13", "R", "X", "XXX"])

In [7]:
# Copy images to temporary directories
for img in train_list:
    shutil.copy(img, os.path.join(dataDir, 'images', 'train', img.split('/')[-2])) 
for img in val_list:
    shutil.copy(img, os.path.join(dataDir, 'images', 'val', img.split('/')[-2]))
for img in test_list:
    shutil.copy(img, os.path.join(dataDir, 'images', 'test', img.split('/')[-2]))

In [8]:
for dir in topdir:
    for i in os.listdir(os.path.join(dataDir, 'images', dir)):
        print('train', i, len(os.listdir(os.path.join(dataDir, 'images', dir, i))))
    print()

train PG 4008
train PG13 4031
train R 4031
train X 4019
train XXX 3994

train PG 501
train PG13 504
train R 504
train X 502
train XXX 499

train PG 502
train PG13 504
train R 504
train X 503
train XXX 500



In [11]:
# Train the model
results = model.train(
    data='../data/may_eval_dataset/images/',
    epochs=100,
    imgsz=224,
     project='../models',  # Directory where the results will be saved
    name='june_yolo_cls'  # Name of the experiment sub-directory
)

Ultralytics YOLOv8.2.45 🚀 Python-3.10.12 torch-2.1.0+cu118 CUDA:0 (NVIDIA RTX A4000, 16102MiB)
engine/trainer: task=classify, mode=train, model=yolov8n-cls.pt, data=../data/may_eval_dataset/images/, epochs=100, time=None, patience=100, batch=16, imgsz=224, save=True, save_period=-1, cache=False, device=None, workers=8, project=../models, name=june_yolo_cls, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_

100%|██████████| 6.23M/6.23M [00:00<00:00, 107MB/s]


AMP: checks passed ✅


train: Scanning /workspace/data/may_eval_dataset/images/train... 20083 images, 0 corrupt: 100%|██████████| 20083/20083 [00:07<00:00, 2713.04it/s]


train: New cache created: /workspace/data/may_eval_dataset/images/train.cache


...

    100/100      0.51G      0.232          3        224: 100%|██████████| 1256/1256 [01:14<00:00, 16.97it/s]
               classes   top1_acc   top5_acc: 100%|██████████| 79/79 [00:06<00:00, 12.27it/s]

                   all      0.737          1



100 epochs completed in 2.234 hours.
Optimizer stripped from ../models/june_yolo_cls/weights/last.pt, 3.0MB
Optimizer stripped from ../models/june_yolo_cls/weights/best.pt, 3.0MB

Validating ../models/june_yolo_cls/weights/best.pt...
Ultralytics YOLOv8.2.45 🚀 Python-3.10.12 torch-2.1.0+cu118 CUDA:0 (NVIDIA RTX A4000, 16102MiB)
YOLOv8n-cls summary (fused): 73 layers, 1441285 parameters, 0 gradients, 3.3 GFLOPs
train: /workspace/data/may_eval_dataset/images/train... found 20083 images in 5 classes ✅ 
val: /workspace/data/may_eval_dataset/images/val... found 2510 images in 5 classes ✅ 
test: /workspace/data/may_eval_dataset/images/test... found 2513 images in 5 classes ✅ 


               classes   top1_acc   top5_acc: 100%|██████████| 79/79 [00:05<00:00, 13.83it/s]


                   all      0.738          1
Speed: 0.2ms preprocess, 0.6ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to ../models/june_yolo_cls
Results saved to ../models/june_yolo_cls
